In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "nyc_taxi"
GOLD_SCHEMA = "gold"

yellow_trips_table = f"{CATALOG}.{GOLD_SCHEMA}.yellow_trips_gold"
green_trips_table = f"{CATALOG}.{GOLD_SCHEMA}.green_trips_gold"

### Média do valor total `total_amount` recebido em um mês para todos os yellow táxis da frota

In [0]:
df = spark.table(yellow_trips_table)
df_monthly_avg = df \
    .groupBy("year_month") \
    .agg(F.round(F.avg("total_amount"), 2).alias("AVG_TOTAL_AMOUNT")) \
    .orderBy("year_month")

print("\nMédia do valor total por mês - NYC Yellow Taxis \n")
display(df_monthly_avg)


Média do valor total por mês - NYC Yellow Taxis 



year_month,AVG_TOTAL_AMOUNT
2001-01,56.28
2002-12,37.19
2003-01,85.74
2008-12,48.01
2009-01,16.4
2014-11,52.38
2022-10,55.38
2022-12,26.33
2023-01,27.36
2023-02,27.27


### Média de passageiros `passenger_count` por cada hora do dia que pegaram táxi no mês de maio,considerando todos os táxis da frota

> *Interpretação A*: Qual a média da quantidade passageiros (volume) por hora para determinado mês?

In [0]:
df_all_taxis = spark.table(yellow_trips_table).union(spark.table(green_trips_table))

# Step 1: Aggregate total passengers by (day, hour)
df_daily_hourly = df_all_taxis \
    .filter(F.col("year_month") == "2023-05") \
    .withColumn("day_of_month", F.dayofmonth("pickup_datetime")) \
    .groupBy("day_of_month", "pickup_hour") \
    .agg(F.sum("passenger_count").alias("total_passengers_that_day_hour"))

# Step 2: Calculate average across days for each hour
df_daily_avg = df_daily_hourly \
    .groupBy("pickup_hour") \
    .agg(F.round(F.avg("total_passengers_that_day_hour"), 2).alias("AVG_TOTAL_PASSENGERS")) \
    .orderBy("pickup_hour")


print("\nMédia de passageiros por hora do dia para o mês de Maio:\n")

display(df_daily_avg)



Média de passageiros por hora do dia para o mês de maio



pickup_hour,AVG_TOTAL_PASSENGERS
0,4072.58
1,2661.0
2,1733.06
3,1122.52
4,701.61
5,742.1
6,1855.48
7,3839.39
8,5291.39
9,6022.29


> *Interpretação B*: Qual é a média de ocupação do veículo (passageiros por viagem) por hora?

In [0]:
df_all_taxis.filter(F.col("year_month") == "2023-05") \
    .groupBy("pickup_hour") \
    .agg(F.round(F.avg("passenger_count"), 2).alias("AVG_PASSENGER_COUNT")) \
    .orderBy("pickup_hour")

print("\nMédia da ocupação de passageiros por hora do dia para o mês de Maio:\n")

display(df_all_taxis)


Média da ocupação de passageiros por hora do dia para o mês de Maio:



pickup_hour,AVG_PASSENGER_COUNT
0,1.41
1,1.42
2,1.44
3,1.44
4,1.39
5,1.27
6,1.24
7,1.25
8,1.27
9,1.28


## Comparação das Interpretações


**Interpretação A (Volume Total)**
* Valores altos indicam horários de pico de demanda
* Picos esperados: Manhã (7-9h), Fim de tarde (17-19h)
* Baixos esperados: Madrugada (3-5h)
* Grande variação entre pico e vale

**Interpretação B (Ocupação)**
* Valores próximos de 1,0 indicam viagens individuais
* Valores > 1,5 indicam mais viagens em grupo/família
* Variação esperada: Pequena (tipicamente 1,2-1,6)
* Padrão pode ser mais uniforme ao longo do dia

---

### Quando Usar Cada Uma

**Use a Interpretação A quando:**
* Dimensionar a capacidade da frota
* Prever demanda futura
* Planejar recursos operacionais
* Pergunta: "Quantos veículos preciso neste horário?"

**Use a Interpretação B quando:**
* Avaliar a eficiência de utilização dos veículos
* Desenvolver estratégias de ride-sharing
* Analisar padrões de comportamento dos clientes
* Pergunta: "Os veículos estão sendo bem utilizados?"

---

**Nota:** Ambas as interpretações são válidas e complementares. A escolha depende do objetivo de negócio.